# Fine-tune Neha (QLoRA)

Train a small Qwen2.5-1.5B-Instruct model on your Hinglish service-advisor transcripts, then serve it as an OpenAI-compatible API for `human_bot`.

Run on **Google Colab (T4 GPU)** or any GPU pod (RunPod / Lambda / Modal).

Files needed in the same folder:
- `ft_data.jsonl` (starter examples + your exported transcripts)

In [ ]:
# Mount Google Drive (optional, keeps checkpoints after runtime ends)
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/neha_ft

In [ ]:
!pip install -q torch transformers accelerate peft trl datasets bitsandbytes scipy

In [ ]:
import json, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_FILE = "ft_data.jsonl"
OUT_DIR = "./neha-ft"

# Load JSONL
with open(DATA_FILE, encoding="utf-8") as f:
    data = [json.loads(line) for line in f]
dataset = Dataset.from_list(data)
print(f"Loaded {len(dataset)} conversations")
print(dataset[0])

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# 4-bit quantized base model (fits on Colab T4 / 16GB)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=300,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=lambda x: x["messages"],
    args=args,
    tokenizer=tokenizer,
)

trainer.train()
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print(f"Saved adapter to {OUT_DIR}")

In [ ]:
# Quick inference test
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype="auto", device_map="auto", trust_remote_code=True
)
model = PeftModel.from_pretrained(base, OUT_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

msgs = [
    {"role": "system", "content": "You are Neha, a female Auto Car Repair service advisor in Delhi NCR. Speak short, natural Hinglish. Ask one question per turn."},
    {"role": "user", "content": "AC service ka kitna hai?"},
]
text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=80, temperature=0.7)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## Next: merge & serve

Merge the LoRA into the base model and run vLLM as an OpenAI-compatible API:
```bash
pip install vllm
python -m vllm.entrypoints.api_server --model ./neha-ft-merged --port 8000 --host 0.0.0.0
```

Then set on Render / Railway:
```
HUMAN_BOT_LLM_ORDER=openai-compatible
OPENAI_BASE_URL=https://your-endpoint/v1
OPENAI_API_KEY=dummy
HUMAN_BOT_MODEL=neha-ft
```